# Fine-Tuning + Distillation on a Free Colab GPU

Distil a **large open-source teacher** into a **small open-source student** and
QLoRA-fine-tune it on **specialised-domain data** — the whole thing fits a free
Colab **T4 (15 GB)**.

**Runtime → Change runtime type → T4 GPU** before running.

Pipeline:
1. Load a domain instruction dataset from the HF Hub.
2. (Optional) A larger teacher generates responses → distillation targets.
3. QLoRA-fine-tune the 0.5B student to imitate them.
4. Try the fine-tuned model.


## 1. Install & check the GPU

In [ ]:
!nvidia-smi -L
!pip install -q transformers datasets accelerate peft bitsandbytes sentencepiece

## 2. Get the repo code

The reusable pipeline lives in `src/finetune/`. Clone it (or `pip install -e .`).

In [ ]:
import os
if not os.path.exists('model-distillation-'):
    !git clone https://github.com/aabhimittal/model-distillation-
%cd model-distillation-
import sys; sys.path.insert(0, os.getcwd())

## 3. Configure

A 0.5B student trains comfortably on a T4. Swap `data.name` for any Alpaca- or
Q&A-style domain dataset (finance, medical, legal, code…) and remap columns via
the `*_key` fields.

In [ ]:
from src.finetune.config import FinetuneConfig

cfg = FinetuneConfig()
cfg.model.student   = "Qwen/Qwen2.5-0.5B-Instruct"
cfg.data.name       = "gbharti/finance-alpaca"   # <-- your specialised domain
cfg.data.max_train_samples = 800                 # small for a quick first run
cfg.data.max_eval_samples  = 100
cfg.train.num_epochs = 1
cfg.train.output_dir = "outputs/student_finetuned"

# Teacher for distillation:
#   "passthrough" -> fine-tune on the dataset's gold answers (no teacher, fastest)
#   "hf"          -> load Qwen2.5-3B locally and generate (needs more VRAM/time)
#   "nim"         -> NVIDIA NIM API teacher (set NVIDIA_API_KEY; no local GPU cost)
cfg.teacher.provider = "passthrough"
cfg

## 4. Load the domain dataset

In [ ]:
from src.finetune.data import load_domain_dataset, records_to_dicts, train_eval_split

ds = load_domain_dataset(cfg.data)
train_ds, eval_ds = train_eval_split(ds, cfg.data.max_eval_samples)
train_records = records_to_dicts(train_ds)
eval_records  = records_to_dicts(eval_ds)
print(f"train={len(train_records)}  eval={len(eval_records)}")
print("Example:", train_records[0])

## 5. (Optional) Distillation targets from a teacher

With `provider='passthrough'` this is a no-op and we fine-tune on gold answers.
Switch to `'hf'` or `'nim'` to distil a larger teacher's responses instead.

In [ ]:
from src.finetune.distill import build_teacher, make_distillation_records, resolve_provider

if resolve_provider(cfg.teacher) != "passthrough":
    teacher = build_teacher(cfg.teacher, cfg.model.teacher)
    teacher_outputs = teacher.generate(train_records)
    train_records = make_distillation_records(train_records, teacher_outputs)
    print("Distilled", len(teacher_outputs), "teacher responses.")
else:
    print("passthrough: fine-tuning on gold answers.")

## 6. Build the QLoRA student

The base is frozen + quantised to 4-bit (NF4); only small LoRA adapters train —
well under 1% of parameters.

In [ ]:
from src.finetune.model import build_student, count_trainable_parameters

model, tokenizer = build_student(cfg.model, cfg.lora)
trainable, total = count_trainable_parameters(model)
print(f"trainable {trainable:,} / {total:,}  ({100*trainable/total:.3f}%)")

## 7. Fine-tune

In [ ]:
from src.finetune.train import train_student

train_student(cfg, model, tokenizer, train_records, eval_records)
print("Saved adapter ->", cfg.train.output_dir)

## 8. Try the fine-tuned model

In [ ]:
import torch
from src.finetune.chat import to_messages

prompt = "Explain what an ETF is in one sentence."
text = tokenizer.apply_chat_template(to_messages(prompt), tokenize=False, add_generation_prompt=True)
enc = tokenizer(text, return_tensors="pt").to(model.device)
out = model.generate(**enc, max_new_tokens=128, do_sample=False,
                     pad_token_id=tokenizer.pad_token_id or tokenizer.eos_token_id)
print(tokenizer.decode(out[0][enc['input_ids'].shape[1]:], skip_special_tokens=True).strip())

## 9. (Optional) Push the adapter to the HF Hub

```python
from huggingface_hub import login; login()
model.push_to_hub("your-username/qwen2.5-0.5b-finance-lora")
tokenizer.push_to_hub("your-username/qwen2.5-0.5b-finance-lora")
```

The LoRA adapter is only a few MB — a highly efficient, private specialised model.
